# What is Face Recognition?

**Face Recognition** is the task of identifying a person from a face image.

There are two basic settings:

$$
\boxed{\text{Face Verification}}
$$

> **"Is this person who they claim to be?"**

This is a **1-to-1** problem:

$$
\text{Face Image}
+
\text{Claimed Identity}
\rightarrow
\text{Match / Not Match}
$$

For example, verifying whether a face matches the registered user of a phone.

---

$$
\boxed{\text{Face Recognition}}
$$

> **"Who is this person?"**

This is a **1-to-$K$** problem:

$$
\text{Face Image}
\rightarrow
\{\text{Person}_1,\ldots,\text{Person}_K\}
$$

For example, identifying a person from a database of employees.

The core idea is:

$$
\boxed{
\text{Face Image}
\rightarrow
\text{Identity}
}
$$

In deep learning, face recognition is commonly based on learning a useful **face representation (embedding)** rather than treating every identity as a fixed classification class.

# One-Shot Learning

**One-Shot Learning** is a learning setting where a model should be able to recognize a new class or identity using **only one or a very small number of examples**.

In the context of Face Recognition, this is important because a new person may need to be added to the system without retraining the entire model.

---

## 1. The Problem with Conventional Classification

Suppose a classifier is trained with:

$$
K=100
$$

people:

$$
\text{Image}
\rightarrow
\text{Neural Network}
\rightarrow
\text{Softmax}
\rightarrow
\{\text{Person}_1,\ldots,\text{Person}_{100}\}
$$

If a new person appears:

$$
\text{Person}_{101}
$$

the original classifier does not have a representation for this new identity.

This can require:

- adding a new output class;
- collecting training data;
- retraining or fine-tuning the classifier.

This is undesirable when the set of identities changes frequently.

---

## 2. The Core Idea of One-Shot Learning

Instead of directly learning:

$$
\text{Image}
\rightarrow
\text{Identity}
$$

we learn a representation:

$$
\boxed{
\text{Image}
\rightarrow
\text{Embedding}
}
$$

Let:

$$
f(x)
$$

be the learned embedding function.

The goal is to make embeddings of the same identity close together:

$$
f(x_A^{(1)})
\approx
f(x_A^{(2)})
$$

while embeddings of different identities are farther apart:

$$
f(x_A)
\not\approx
f(x_B)
$$

Therefore:

$$
\boxed{
\text{Same Identity}
\rightarrow
\text{Similar Embeddings}
}
$$

and:

$$
\boxed{
\text{Different Identities}
\rightarrow
\text{Different Embeddings}
}
$$

---

## 3. Face Recognition Example

Suppose the database contains only one image for each person:

```text
Alice   → Face Image A
Bob     → Face Image B
Charlie → Face Image C
```

Given a query image:

```text
Query Face
```

the system computes:

$$
f(x_q)
$$

and compares it with:

$$
f(x_A),f(x_B),f(x_C)
$$

For example, using Euclidean distance:

$$
d(f(x_q),f(x_A))
$$

$$
d(f(x_q),f(x_B))
$$

$$
d(f(x_q),f(x_C))
$$

Suppose:

$$
d(f(x_q),f(x_B))
$$

is the smallest and satisfies:

$$
d\le\tau
$$

Then the system concludes:

$$
\boxed{
x_q=\text{Bob}
}
$$

---

## 4. Why This Is Useful

The database can grow:

$$
K
\rightarrow
K+1
\rightarrow
K+2
\rightarrow
\cdots
$$

without necessarily retraining the embedding network.

Instead, a new identity can simply be represented by its embedding:

$$
f(x_{\text{new}})
$$

and stored in the database.

Thus:

$$
\boxed{
\text{New Identity}
\rightarrow
\text{Add Example / Embedding}
}
$$

rather than:

$$
\boxed{
\text{New Identity}
\rightarrow
\text{Retrain the Entire Classifier}
}
$$

---

## 5. Important Clarification

**One-Shot Learning does not necessarily mean that the model itself is trained using exactly one example.**

In the Face Recognition setting, the important idea is:

> **After learning a general representation, the model can recognize a previously unseen identity from one or very few examples without retraining the entire model.**

Thus the overall idea is:

$$
\boxed{
\text{Learn a General Representation}
\rightarrow
\text{Compare New Examples}
}
$$

rather than:

$$
\boxed{
\text{Train a New Classifier for Every New Identity}
}
$$

This motivates the next step in the course:

$$
\boxed{
\text{One-Shot Learning}
\rightarrow
\text{Similarity Learning}
\rightarrow
\text{Siamese Network}
}
$$

# Siamese Network

**Siamese Network** is an architecture designed to learn the **similarity between two inputs**.

In Face Recognition, instead of directly learning:

$$
\text{Face Image}
\rightarrow
\text{Person ID}
$$

the network learns:

$$
\boxed{
\text{Face Image}_1
+
\text{Face Image}_2
\rightarrow
\text{Similarity}
}
$$

The core idea is to process both inputs using the **same network with shared parameters**, transform them into embeddings, and then compare those embeddings.

---

## 1. Siamese Network Architecture

A Siamese Network contains two branches with shared parameters:

```mermaid
flowchart LR
    A["Image 1"] --> B["Shared CNN"]
    C["Image 2"] --> D["Shared CNN"]

    B --> E["Embedding f(x₁)"]
    D --> F["Embedding f(x₂)"]

    E --> G["Distance / Similarity"]
    F --> G

    G --> H["Similarity Decision"]
```

Both branches use the same function:

$$
f(x;\theta)
$$

Therefore:

$$
f(x_1;\theta)
$$

and:

$$
f(x_2;\theta)
$$

use exactly the same parameter set:

$$
\boxed{
\theta_1=\theta_2=\theta
}
$$

This is the meaning of **shared parameters**.

---

## 2. Why Are Parameters Shared?

Suppose the two branches were completely independent:

$$
f_1(x_1;\theta_1)
$$

and:

$$
f_2(x_2;\theta_2)
$$

Then the two networks might learn different representation spaces, making their outputs difficult to compare meaningfully.

Siamese Network instead enforces:

$$
\boxed{
f(x_1;\theta)
\quad\text{and}\quad
f(x_2;\theta)
}
$$

to be produced by the same transformation.

Therefore both embeddings lie in the same representation space.

This allows us to compute:

$$
d(f(x_1),f(x_2))
$$

meaningfully.

---

## 3. Embedding

Each branch converts the input into an embedding:

$$
f(x)\in\mathbb R^d
$$

For example:

$$
f(x_1)
=
\begin{bmatrix}
0.2\\
-0.5\\
0.8
\end{bmatrix}
$$

and:

$$
f(x_2)
=
\begin{bmatrix}
0.3\\
-0.4\\
0.7
\end{bmatrix}
$$

These vectors represent the two inputs in the same embedding space.

The individual dimensions do not necessarily correspond to clearly interpretable properties.

What matters is the **relationship between embeddings**.

---

## 4. Measuring Similarity

A simple approach is Euclidean distance:

$$
d(x_1,x_2)
=
\|f(x_1)-f(x_2)\|_2
$$

or:

$$
d(x_1,x_2)
=
\sqrt{
\sum_{i=1}^{d}
\left(
f_i(x_1)-f_i(x_2)
\right)^2
}
$$

If the inputs represent the same person, we want:

$$
d(x_1,x_2)\approx0
$$

If they represent different people, we want the distance to be larger.

We can then use a threshold:

$$
d(x_1,x_2)\le\tau
\Rightarrow
\text{Same Person}
$$

and:

$$
d(x_1,x_2)>\tau
\Rightarrow
\text{Different People}
$$

Thus the network learns a geometry where:

$$
\boxed{
\text{Same Identity}
\rightarrow
\text{Close Embeddings}
}
$$

and:

$$
\boxed{
\text{Different Identity}
\rightarrow
\text{Far Embeddings}
}
$$

---

## 5. Training Data

A Siamese Network can be trained using pairs:

$$
(x_1,x_2,y)
$$

where:

$$
y\in\{0,1\}
$$

with:

$$
y=1
$$

for the same identity, and:

$$
y=0
$$

for different identities.

For example:

```text
Image 1        Image 2        Label

Alice          Alice            1
Alice          Bob              0
Bob            Bob              1
Bob            Charlie          0
```

The desired behavior is:

$$
y=1
\rightarrow
d(f(x_1),f(x_2))\text{ small}
$$

and:

$$
y=0
\rightarrow
d(f(x_1),f(x_2))\text{ large}
$$

---

## 6. Contrastive Loss

A natural loss for this setting is **contrastive loss**:

$$
L
=
y\,d^2
+
(1-y)\max(0,m-d)^2
$$

where:

- $d=d(f(x_1),f(x_2))$;
- $y=1$ means same identity;
- $y=0$ means different identities;
- $m$ is the margin.

### Same Identity

If:

$$
y=1
$$

then:

$$
L=d^2
$$

To minimize the loss:

$$
d\rightarrow0
$$

so the embeddings are pulled together.

### Different Identity

If:

$$
y=0
$$

then:

$$
L=\max(0,m-d)^2
$$

If:

$$
d\ge m
$$

then:

$$
L=0
$$

Therefore, embeddings of different identities only need to be separated by at least the margin $m$.

The resulting geometry is:

$$
\boxed{
\text{Same Identity}
\rightarrow
\text{Pull Together}
}
$$

$$
\boxed{
\text{Different Identity}
\rightarrow
\text{Push Apart}
}
$$

---

## 7. Forward Propagation

Given a pair:

$$
(x_1,x_2)
$$

the two branches compute:

$$
a_1=f(x_1;\theta)
$$

and:

$$
a_2=f(x_2;\theta)
$$

Then:

$$
d=\|a_1-a_2\|_2
$$

and:

$$
L=L(d,y)
$$

The complete forward process is:

```mermaid
flowchart LR
    A["x₁"] --> B["Shared Network f(·; θ)"]
    C["x₂"] --> D["Shared Network f(·; θ)"]

    B --> E["Embedding a₁"]
    D --> F["Embedding a₂"]

    E --> G["Distance d"]
    F --> G

    G --> H["Loss L"]
```

The important point is:

$$
\boxed{
\text{The two branches share the same parameters}
}
$$

They are not two independently trained networks.

---

## 8. Backward Propagation

This is an important consequence of parameter sharing.

The loss produces gradients with respect to both embeddings:

$$
\frac{\partial L}{\partial a_1}
$$

and:

$$
\frac{\partial L}{\partial a_2}
$$

These gradients then propagate through the two branches.

Branch 1 contributes:

$$
\frac{\partial L}{\partial\theta}\bigg|_1
$$

Branch 2 contributes:

$$
\frac{\partial L}{\partial\theta}\bigg|_2
$$

Since both branches use the same parameters $\theta$, the total gradient is:

$$
\boxed{
\frac{\partial L}{\partial\theta}
=
\frac{\partial L}{\partial\theta}\bigg|_1
+
\frac{\partial L}{\partial\theta}\bigg|_2
}
$$

The parameter update is then:

$$
\theta^{new}
=
\theta^{old}
-
\eta
\frac{\partial L}{\partial\theta}
$$

This is the same principle of shared-parameter gradient accumulation that appears in other architectures with shared computation.

---

## 9. Same Person: Gradient Intuition

Suppose:

$$
y=1
$$

but the two embeddings are too far apart:

$$
d\gg0
$$

The loss:

$$
L=d^2
$$

creates gradients that move the embeddings closer together.

Conceptually:

```text
Before training

Embedding 1 ●                       ● Embedding 2


After optimization

Embedding 1 ●           ● Embedding 2
```

Because the network parameters are shared, changing $\theta$ affects how **both types of faces** are represented.

The network therefore learns a transformation that makes same-identity examples naturally closer.

---

## 10. Different People: Gradient Intuition

Suppose:

$$
y=0
$$

and:

$$
d<m
$$

Then:

$$
L=(m-d)^2
$$

The network receives a gradient that pushes the two embeddings farther apart.

Once:

$$
d\ge m
$$

we have:

$$
L=0
$$

and that pair no longer contributes a positive separation force.

Therefore the network learns:

$$
\boxed{
\text{Same Person}
\rightarrow
\text{Cluster Together}
}
$$

and:

$$
\boxed{
\text{Different People}
\rightarrow
\text{Separate}
}
$$

---

## 11. Siamese Network and One-Shot Learning

This directly connects to **One-Shot Learning**.

Instead of learning:

$$
\text{Image}
\rightarrow
\text{Fixed Identity Class}
$$

the network learns:

$$
\text{Image}
\rightarrow
\text{General Embedding}
$$

After training, a new identity can be represented by an embedding without creating a new classifier class.

For a new person:

$$
x_{\text{new}}
\rightarrow
f(x_{\text{new}})
$$

The embedding can then be stored in a database.

For a query image:

$$
x_q
\rightarrow
f(x_q)
$$

and compared against stored embeddings.

Thus:

$$
\boxed{
\text{Learn Representation Once}
\rightarrow
\text{Compare New Identities Later}
}
$$

---

## 12. Siamese Network for Face Verification

A simple verification system works as follows:

```mermaid
flowchart LR
    A["Query Face"] --> B["Shared CNN"]
    C["Registered Face"] --> D["Shared CNN"]

    B --> E["Embedding 1"]
    D --> F["Embedding 2"]

    E --> G["Distance"]
    F --> G

    G --> H{"d ≤ τ?"}
    H -->|Yes| I["Same Person"]
    H -->|No| J["Different Person"]
```

The system therefore reduces Face Verification to:

$$
\boxed{
\text{Embedding}
\rightarrow
\text{Distance}
\rightarrow
\text{Threshold}
}
$$

---

## 13. Siamese Network vs. Conventional Classification

### Conventional Classification

$$
\text{Image}
\rightarrow
\text{Feature}
\rightarrow
\text{Softmax}
\rightarrow
\text{Fixed Identity Class}
$$

The classifier is tied to the identities represented during training.

### Siamese Network

$$
\text{Image}_1
\rightarrow
\text{Embedding}_1
$$

$$
\text{Image}_2
\rightarrow
\text{Embedding}_2
$$

then:

$$
\text{Embedding}_1
\leftrightarrow
\text{Embedding}_2
$$

The model learns a **comparison space**, rather than requiring a fixed output class for every identity.

Therefore:

$$
\boxed{
\text{Classification}
=
\text{Which Known Class?}
}
$$

while:

$$
\boxed{
\text{Siamese Network}
=
\text{How Similar Are These Two Inputs?}
}
$$

---

## 14. Core Mental Model

The complete Siamese Network can be remembered as:

$$
\boxed{
\text{Two Inputs}
}
$$

↓

$$
\boxed{
\text{Same Shared Network}
}
$$

↓

$$
\boxed{
\text{Two Embeddings}
}
$$

↓

$$
\boxed{
\text{Distance / Similarity}
}
$$

↓

$$
\boxed{
\text{Same or Different}
}
$$

During training:

$$
\boxed{
\text{Same Identity}
\rightarrow
\text{Pull Embeddings Together}
}
$$

$$
\boxed{
\text{Different Identity}
\rightarrow
\text{Push Embeddings Apart}
}
$$

Because both branches share $\theta$:

$$
\boxed{
\frac{\partial L}{\partial\theta}
=
\frac{\partial L}{\partial\theta}\bigg|_1
+
\frac{\partial L}{\partial\theta}\bigg|_2
}
$$

The most important insight is:

> **A Siamese Network is not two independent neural networks. It is the same neural network applied to two inputs, with shared parameters, so that the resulting embeddings lie in a common representation space where similarity can be learned and measured.**

This idea leads naturally to the next major concept in Week 4:

$$
\boxed{
\text{Siamese Network}
\rightarrow
\text{Triplet Loss}
}
$$

# Triplet Loss

**Triplet Loss** is a loss function used to learn a meaningful **embedding space**, especially for Face Recognition.

Instead of only comparing two images, Triplet Loss considers three images:

$$
\boxed{
\text{Anchor}+\text{Positive}+\text{Negative}
}
$$

The objective is:

> **The Anchor should be closer to the Positive than to the Negative by at least a margin $\alpha$.**

---

## 1. The Triplet

A triplet consists of:

- **Anchor ($A$)**: an image of a person.
- **Positive ($P$)**: another image of the **same person** as the Anchor.
- **Negative ($N$)**: an image of a **different person**.

For example:

```text
A → Alice image 1
P → Alice image 2
N → Bob image 1
```

The desired relationship is:

$$
d(A,P)<d(A,N)
$$

But we want a stronger constraint:

$$
\boxed{
d(A,P)+\alpha\le d(A,N)
}
$$

where:

$$
\alpha>0
$$

is the **margin**.

---

## 2. Shared Embedding Network

The three images are passed through the **same network**:

$$
f(x;\theta)
$$

to obtain three embeddings:

$$
a=f(A;\theta)
$$

$$
p=f(P;\theta)
$$

$$
n=f(N;\theta)
$$

Conceptually:

```mermaid
flowchart LR
    A["Anchor A"] --> B["Shared Network f(·; θ)"]
    P["Positive P"] --> C["Shared Network f(·; θ)"]
    N["Negative N"] --> D["Shared Network f(·; θ)"]

    B --> E["Embedding a"]
    C --> F["Embedding p"]
    D --> G["Embedding n"]

    E --> H["Triplet Loss"]
    F --> H
    G --> H
```

The three branches are not three independent networks.

They are three uses of the same function:

$$
\boxed{
\theta_A=\theta_P=\theta_N=\theta
}
$$

---

## 3. Distance in the Embedding Space

A common choice is squared Euclidean distance:

$$
d(A,P)=\|a-p\|_2^2
$$

and:

$$
d(A,N)=\|a-n\|_2^2
$$

The model therefore learns the geometry of the embedding space:

$$
\boxed{
\text{Same Identity}
\rightarrow
\text{Close Embeddings}
}
$$

and:

$$
\boxed{
\text{Different Identity}
\rightarrow
\text{Far Embeddings}
}
$$

---

## 4. Triplet Loss Function

The Triplet Loss is:

$$
\boxed{
L(A,P,N)
=
\max
\left(
\|a-p\|_2^2
-
\|a-n\|_2^2
+
\alpha,
0
\right)
}
$$

Using the distance notation:

$$
\boxed{
L=
\max
\left(
d(A,P)-d(A,N)+\alpha,
0
\right)
}
$$

The desired condition is:

$$
\boxed{
d(A,P)+\alpha\le d(A,N)
}
$$

---

## 5. Why Is the Margin Necessary?

Suppose we only required:

$$
d(A,P)<d(A,N)
$$

Then the difference could be extremely small.

For example:

$$
d(A,P)=0.49
$$

and:

$$
d(A,N)=0.50
$$

Although the condition is technically satisfied, the separation is not meaningful.

The margin requires:

$$
d(A,N)-d(A,P)\ge\alpha
$$

so the model learns a stronger separation between positive and negative examples.

---

## 6. Example of Triplet Loss

Suppose:

$$
d(A,P)=0.2
$$

$$
d(A,N)=0.8
$$

and:

$$
\alpha=0.2
$$

Then:

$$
L=
\max(0.2-0.8+0.2,0)
$$

$$
=
\max(-0.4,0)
=
0
$$

The triplet already satisfies:

$$
0.2+0.2\le0.8
$$

so no additional separation is required.

---

Now suppose:

$$
d(A,P)=0.5
$$

$$
d(A,N)=0.6
$$

and:

$$
\alpha=0.2
$$

Then:

$$
L=
\max(0.5-0.6+0.2,0)
$$

$$
=0.1
$$

Since:

$$
L>0
$$

the triplet violates the desired margin.

The network is therefore encouraged to:

$$
\boxed{
d(A,P)\downarrow
}
$$

and/or:

$$
\boxed{
d(A,N)\uparrow
}
$$

until:

$$
d(A,N)\ge d(A,P)+\alpha
$$

---

## 7. Forward Propagation

For one triplet:

$$
(A,P,N)
$$

the network computes:

$$
a=f(A;\theta)
$$

$$
p=f(P;\theta)
$$

$$
n=f(N;\theta)
$$

Then:

$$
d_{AP}=\|a-p\|_2^2
$$

$$
d_{AN}=\|a-n\|_2^2
$$

and finally:

$$
L=
\max(d_{AP}-d_{AN}+\alpha,0)
$$

Thus the forward flow is:

$$
\boxed{
(A,P,N)
\rightarrow
(a,p,n)
\rightarrow
(d_{AP},d_{AN})
\rightarrow
L
}
$$

---

## 8. Backward Propagation

If:

$$
L=0
$$

then:

$$
d_{AP}+\alpha\le d_{AN}
$$

and this triplet does not need further separation.

If:

$$
L>0
$$

the gradient propagates through all three embeddings:

$$
\frac{\partial L}{\partial a},
\qquad
\frac{\partial L}{\partial p},
\qquad
\frac{\partial L}{\partial n}
$$

These gradients then propagate through the shared network.

The three computational paths contribute to the same parameter set:

$$
\boxed{
\frac{\partial L}{\partial\theta}
=
\frac{\partial L}{\partial\theta}\bigg|_A
+
\frac{\partial L}{\partial\theta}\bigg|_P
+
\frac{\partial L}{\partial\theta}\bigg|_N
}
$$

The update is:

$$
\theta^{new}
=
\theta^{old}
-
\eta
\frac{\partial L}{\partial\theta}
$$

Therefore, just as in a Siamese Network, **the three branches share parameters and their gradient contributions are accumulated into one update**.

---

## 9. Triplet Loss as Relative Learning

Contrastive Loss can ask:

> Are these two inputs similar or different?

Triplet Loss asks a stronger question:

> **Is the Positive closer to the Anchor than the Negative by at least a margin?**

Thus Triplet Loss learns a **relative relationship**:

$$
\boxed{
d(A,P)+\alpha\le d(A,N)
}
$$

rather than simply learning whether a pair is similar.

This is why Triplet Loss is a form of **metric learning**.

---

## 10. Embedding-Space Intuition

After training, examples of the same identity should form clusters:

```text
Alice:    ● ● ●

Bob:                    ● ● ●

Charlie:                            ● ● ●
```

Conceptually:

$$
\boxed{
\text{Same Identity}
\rightarrow
\text{Cluster Together}
}
$$

while:

$$
\boxed{
\text{Different Identities}
\rightarrow
\text{Separated Clusters}
}
$$

The network is therefore learning a representation space in which distance carries semantic meaning.

---

## 11. Connection to One-Shot Face Recognition

Once a good embedding function is learned:

$$
f(x;\theta)
$$

a new identity does not necessarily require retraining the network.

For example, a new person can be represented by:

$$
x_{\text{new}}
\rightarrow
f(x_{\text{new}})
$$

and the embedding can be stored.

For a query image:

$$
x_q
\rightarrow
f(x_q)
$$

we compare:

$$
d(f(x_q),f(x_{\text{new}}))
$$

with a threshold.

Thus:

$$
\boxed{
\text{Learn Representation Once}
\rightarrow
\text{Compare New Identities Later}
}
$$

This is the connection between:

$$
\boxed{
\text{One-Shot Learning}
\rightarrow
\text{Siamese Network}
\rightarrow
\text{Triplet Loss}
}
$$

---

## 12. Core Mental Model

Triplet Loss can be remembered with one sentence:

> **Make the Anchor closer to the Positive than to the Negative by at least a margin $\alpha$.**

Mathematically:

$$
\boxed{
d(A,P)+\alpha\le d(A,N)
}
$$

with loss:

$$
\boxed{
L=
\max
\left(
d(A,P)-d(A,N)+\alpha,
0
\right)
}
$$

where:

$$
\boxed{
A=\text{Anchor}
}
$$

$$
\boxed{
P=\text{Positive: same identity}
}
$$

$$
\boxed{
N=\text{Negative: different identity}
}
$$

and all three are processed by:

$$
\boxed{
\text{The Same Shared Network }f(x;\theta)
}
$$

The fundamental learning objective is:

$$
\boxed{
\text{Same Identity}
\rightarrow
\text{Close}
}
$$

$$
\boxed{
\text{Different Identity}
\rightarrow
\text{Far}
}
$$

This turns Face Recognition from a fixed-class classification problem into a **metric learning problem over an embedding space**.

# Face Verification and Binary Classification

**Face Verification** is the task of determining whether two face images belong to the **same person**.

It can therefore be formulated as a **binary classification problem**:

$$
\boxed{
y\in\{0,1\}
}
$$

where:

$$
y=
\begin{cases}
1 & \text{same person}\\
0 & \text{different people}
\end{cases}
$$

---

## 1. Face Verification as Binary Classification

Given a pair of face images:

$$
(x_1,x_2)
$$

the model predicts whether they belong to the same identity:

$$
g(x_1,x_2;\theta)
\rightarrow
\hat p
$$

where:

$$
\hat p=P(y=1\mid x_1,x_2)
$$

A threshold can then be used for the final decision:

$$
\hat p\ge\tau
\Rightarrow
\text{Same Person}
$$

$$
\hat p<\tau
\Rightarrow
\text{Different People}
$$

For example:

```text
Face 1              Face 2

┌──────────┐        ┌──────────┐
│          │        │          │
│  Person A│        │  Person A│
│          │        │          │
└──────────┘        └──────────┘

             ↓

        Same Person
```

or:

```text
Face 1              Face 2

┌──────────┐        ┌──────────┐
│          │        │          │
│  Person A│        │  Person B│
│          │        │          │
└──────────┘        └──────────┘

             ↓

      Different People
```

Thus:

$$
\boxed{
\text{Face Verification}
=
\text{1-to-1 Binary Decision}
}
$$

---

## 2. Direct Binary Classification

A straightforward approach is to build a network that directly takes two face images and predicts the probability that they belong to the same person.

Conceptually:

```mermaid
flowchart LR
    A["Face Image 1"] --> C["Pairwise Network"]
    B["Face Image 2"] --> C
    C --> D["P(Same Person)"]
    D --> E["Binary Decision"]
```

The model learns:

$$
(x_1,x_2)
\rightarrow
P(y=1\mid x_1,x_2)
$$

A common binary classification objective is **Binary Cross-Entropy**:

$$
L
=
-
\left[
y\log\hat p
+
(1-y)\log(1-\hat p)
\right]
$$

When:

$$
y=1
$$

the model is encouraged to produce:

$$
\hat p\rightarrow1
$$

When:

$$
y=0
$$

the model is encouraged to produce:

$$
\hat p\rightarrow0
$$

---

## 3. Embedding-Based Verification

A more general approach is to first transform each face into an embedding:

$$
f(x;\theta)
$$

so that:

$$
x_1\rightarrow f(x_1)
$$

and:

$$
x_2\rightarrow f(x_2)
$$

The two embeddings are then compared.

For example, using Euclidean distance:

$$
d
=
\|f(x_1)-f(x_2)\|_2
$$

The verification decision becomes:

$$
d\le\tau
\Rightarrow
\text{Same Person}
$$

and:

$$
d>\tau
\Rightarrow
\text{Different People}
$$

Conceptually:

```mermaid
flowchart LR
    A["Face 1"] --> B["Shared Network"]
    C["Face 2"] --> D["Shared Network"]

    B --> E["Embedding 1"]
    D --> F["Embedding 2"]

    E --> G["Distance"]
    F --> G

    G --> H["Threshold"]
    H --> I["Same / Different"]
```

This formulation leads directly to the **Siamese Network** approach.

---

## 4. Why Embedding-Based Verification Is Useful

Suppose a system contains:

$$
K
$$

identities.

A conventional classifier learns:

$$
\text{Face}
\rightarrow
\text{Softmax}
\rightarrow
\{\text{Person}_1,\ldots,\text{Person}_K\}
$$

The model is therefore tied to a fixed set of identities.

An embedding-based system instead learns:

$$
\text{Face}
\rightarrow
f(x)
$$

When a new person is added, we can store their embedding:

$$
f(x_{\text{new}})
$$

without necessarily retraining the entire network.

Thus:

$$
\boxed{
\text{Learn a General Representation}
\rightarrow
\text{Compare Identities Later}
}
$$

rather than:

$$
\boxed{
\text{Create a New Classification Class}
\rightarrow
\text{Retrain the Model}
}
$$

---

## 5. Face Verification vs. Face Recognition

These two concepts should be clearly distinguished.

### Face Verification

Question:

> **"Are these two faces the same person?"**

$$
\boxed{
1:1
}
$$

### Face Recognition

Question:

> **"Who is this person among the identities in the database?"**

$$
\boxed{
1:K
}
$$

A recognition system can use a verification mechanism repeatedly:

$$
\text{Query}
\leftrightarrow
\text{Person}_1
$$

$$
\text{Query}
\leftrightarrow
\text{Person}_2
$$

$$
\vdots
$$

and select the identity with the best valid similarity.

---

## 6. Core Mental Model

The essential idea is:

$$
\boxed{
\text{Face Verification}
=
\text{1-to-1 Comparison}
}
$$

It can be formulated as binary classification:

$$
\boxed{
(x_1,x_2)
\rightarrow
\{0,1\}
}
$$

There are two important approaches:

### Direct Binary Classification

$$
(x_1,x_2)
\rightarrow
P(\text{Same})
$$

### Embedding-Based Verification

$$
x_1\rightarrow f(x_1)
$$

$$
x_2\rightarrow f(x_2)
$$

$$
f(x_1)\leftrightarrow f(x_2)
\rightarrow
\text{Similarity}
\rightarrow
\{0,1\}
$$

The key distinction is:

> **Face Verification is a 1-to-1 comparison problem: determine whether two face images belong to the same identity. Face Recognition is a 1-to-$K$ identification problem: determine which identity in a database corresponds to a query face.**